# 7 - GRU Hyperparameter Tuning and Training

Part of **RunSight** - see the repo README.

**Purpose.** Builds a two-layer GRU binary classifier (aerobic vs. anaerobic training run), searches its hyperparameters with `keras_tuner` (`RandomSearch`, 10 trials, objective `val_auc`), retrains the best configuration, evaluates it on the test set, and saves the model.

**Inputs**
- `artifacts/gru_arrays.npz` (scaled train/test sequences, labels and continuous scores; written by `6_gru_preprocess.ipynb`)

**Outputs**
- `artifacts/kt_search/` (keras_tuner trial directory; the search resumes from it if re-run)
- `artifacts/gru_model.keras` (trained model; used by `9_gru_evaluate.ipynb`)
- Console output: class weights, best hyperparameters, test classification report, test AUC and confusion matrix

# GRU Model — Hyperparameter Search and Training

**What does this notebook do?**
It reads the tensors prepared by `6_gru_preprocess.ipynb` (`gru_arrays.npz`), builds a GRU (Gated Recurrent Unit) model, searches for the best hyperparameter combination with `keras_tuner`, retrains the model from scratch with that combination, and evaluates it on the test set.

**Why a separate notebook rather than a continuation of `6_gru_preprocess.ipynb`?**
When preprocessing (data alignment, PCA, scaling) and model training (the keras_tuner search) run in the same kernel, RAM usage accumulates: large pandas dataframes plus TensorFlow's own memory usage can crash the kernel (this is exactly what happened in this project). This notebook is self-contained: it does not need the kernel of `6_gru_preprocess.ipynb` and reads everything from disk (`gru_arrays.npz`). Correct order: run `6_gru_preprocess.ipynb` first and save the arrays, shut down that kernel, then run this notebook.

**What happens if training is interrupted?**
The `tuner` object writes every trial to disk under `directory="artifacts/kt_search"`. If the search cell is re-run WITHOUT `overwrite=True`, keras_tuner does not re-run trials that already completed and resumes where it stopped, so a kernel crash does not lose the trials completed so far.

In [ ]:
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from keras_tuner.tuners import RandomSearch

random.seed(46)
np.random.seed(46)
tf.random.set_seed(46)

## 1. Load the preprocessed arrays

**Why not recompute `X_seq` and read it directly from the `.npz` instead?**
`6_gru_preprocess.ipynb` already ran the expensive steps (data alignment, PCA, scaling) once and wrote the result to disk. Repeating them here would waste time and would carry that notebook's entire memory load into this kernel. Saving the arrays as `float32` (instead of float64) also halves the load size.

In [ ]:
data = np.load('artifacts/gru_arrays.npz')
X_train_scaled = data["X_train_scaled"]
X_test_scaled = data["X_test_scaled"]
y_train = data["y_train"]
y_test = data["y_test"]
yc_train = data["yc_train"]
yc_test = data["yc_test"]

print("X_train_scaled:", X_train_scaled.shape, "X_test_scaled:", X_test_scaled.shape)

## 2. Model definition

**Why this architecture: two GRU layers + dropout + a single sigmoid output?**
The first GRU layer (`return_sequences=True`) processes the raw 500-step time series and produces a representation for every step; the second GRU layer compresses that sequence into a single summary vector. A single-neuron `Dense` layer with sigmoid activation then turns this summary into a probability in [0, 1] (aerobic/anaerobic). Two layers were chosen to strike a capacity balance: not large enough to memorize a small dataset, but not so small that it cannot learn.

**Why is dropout placed right after each GRU layer?**
Dropout randomly zeroes a fraction of a layer's output neurons, which prevents the model from relying too heavily on particular neurons (overfitting). Placing it right after each GRU layer forces the next layer (or the output layer) to learn to rely on a random subset of the previous layer's outputs rather than on ALL of them.

**What are `hp.Int`, `hp.Float` and `hp.Choice` for?**
They are not fixed values; they are the way to tell `keras_tuner` "try this parameter in this range / among these options". Which ranges were chosen, and why, is explained in the next section.

**Why `binary_crossentropy` as the loss, and why AUC as a metric (not just accuracy)?**
Since this is a two-class classification problem, `binary_crossentropy` is the natural choice. `AUC` is tracked alongside `accuracy` because the dataset may be class-imbalanced; in that case accuracy can be misleading (a model that always predicts the majority class gets high accuracy), whereas AUC is threshold-independent and measures whether the model can actually separate the two classes.

In [ ]:
def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(X_train_scaled.shape[1], X_train_scaled.shape[2])))
    model.add(GRU(
        units=hp.Int("units_1", min_value=16, max_value=64, step=16),
        return_sequences=True,
        kernel_regularizer=l2(hp.Choice("l2_1", [0.0, 1e-4, 1e-3])),
    ))
    model.add(Dropout(hp.Float("dropout_1", 0.1, 0.5, step=0.1)))
    model.add(GRU(units=hp.Int("units_2", min_value=8, max_value=32, step=8)))
    model.add(Dropout(hp.Float("dropout_2", 0.1, 0.5, step=0.1)))
    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=Adam(learning_rate=hp.Choice("lr", [1e-2, 1e-3, 1e-4])),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return model

## 3. Hyperparameter search (`tuner.search`)

**Why is `class_weight` computed?**
If the two classes in `y_train` (aerobic/anaerobic) are not equally frequent, the model may by default be biased toward the majority class. `class_weight` balances this tendency by assigning a larger penalty weight to each error on the minority class. It is not a data-quality fix; it is a parameter that directly affects the training process.

**Why `objective="val_auc"` and not `val_accuracy`?**
Same reasoning: since accuracy can be misleading under class imbalance, we want the combination the tuner calls "best" to be genuinely good at separating the two classes, not merely at memorizing the majority class.

**How was the search space (units, dropout, l2, learning rate) chosen?**
On a small dataset a large GRU (e.g. 128+ units) would be very prone to memorization, so the layer sizes (16-64 and 8-32) were kept small; a "funnel" architecture was targeted, with the second layer smaller than the first. The dropout range (0.1-0.5) covers the region between light and aggressive regularization. The learning rates ([1e-2, 1e-3, 1e-4]) were chosen on a logarithmic scale because the behavior of the Adam optimizer is sensitive to the order of magnitude of the learning rate, not to its linear difference.

**Why is there an `EarlyStopping`, and what does `patience=5` mean?**
If `val_auc` does not improve for 5 epochs during training, training is stopped early and the best weights are restored (`restore_best_weights=True`). This saves time and keeps the model from starting to memorize in excess epochs.

**Why is `max_trials=10` so small?**
Since each trial needs its own multi-epoch training run (in this project the whole search took about 5.5 hours), 10 trials is a balance between trying a reasonable number of combinations within limited time and practical run time.

In [ ]:
n0, n1 = np.bincount(y_train)
class_weight = {0: len(y_train) / (2 * n0), 1: len(y_train) / (2 * n1)}
print("class_weight:", class_weight)

early_stop = EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True)

tuner = RandomSearch(
    build_model,
    objective="val_auc",
    max_trials=10,
    executions_per_trial=1,
    directory="artifacts/kt_search",
    project_name="gru_aerobic_anaerobic",
    seed=46,
)

tuner.search(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1,
)

## 4. Train and evaluate the best model

**`tuner.search` already trained the models, so why train again?**
`tuner.search` runs each combination for a short number of epochs (here 30) ONLY to compare them with each other; the goal is to find the best one, not to use that model as the final one. Once the best hyperparameters are determined, we train that architecture from scratch for longer (50 epochs, still protected by `EarlyStopping`) to obtain the model that will actually be used.

**Why `gc.collect()` and `tf.keras.backend.clear_session()` here?**
During the search, 10 different models are built and trained back to back, so we clear the memory residue that accumulates in TensorFlow's internal computation graph; otherwise the final training would start by using more RAM than necessary.

In [ ]:
gc.collect()
tf.keras.backend.clear_session()

best_hp = tuner.get_best_hyperparameters(1)[0]
print("best hyperparameters:", best_hp.values)

best_model = tuner.hypermodel.build(best_hp)
history = best_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1,
)

## 5. Evaluation on the test set

**Why look at `roc_auc_score` and the confusion matrix, not just `classification_report`?**
`classification_report` shows precision/recall/f1 per class, which is needed to see which class the model is weaker on. `roc_auc_score` verifies that the `val_auc` metric tracked during training also holds on test data the model has never seen. The confusion matrix shows the actual error distribution behind the numeric summary (how many aerobic runs were taken for anaerobic, and how many anaerobic for aerobic), which is useful for deciding whether the two kinds of error (false positive / false negative) are equally important in this project.

**Why a threshold of 0.5?**
`y_pred_proba >= 0.5` is the most natural default way to turn the model's probability into a binary decision. Since `class_weight` already compensated for class imbalance during training, there is no need to shift the threshold at prediction time (e.g. to 0.4 or 0.6); but if misclassifying one class is more costly than the other, this threshold can be tuned afterwards.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_pred_proba = best_model.predict(X_test_scaled).ravel()
y_pred = (y_pred_proba >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=["aerobic", "anaerobic"]))
print("test AUC:", roc_auc_score(y_test, y_pred_proba))
print("confusion matrix:\n", confusion_matrix(y_test, y_pred))

## 6. Comparison with the baseline

**Isn't `training_zone` (i.e. `y_test`) the output of `run_zone_pipeline` itself? Then what does comparing the GRU against it show?**
An important point to note here: `y_test` is the very LABEL the GRU was trained on, so measuring the GRU's prediction against `y_test` as "accuracy" has already been done (the `classification_report` in the previous section). Building the baseline as a separate model and comparing it against the same label would be a tautology (the threshold logic is the label itself).

**So what is the real question here?**
The GRU was never given the person's personal heart-rate threshold (Karvonen HR reserve ratio); it only saw the raw speed/heart_rate/altitude/GPS channels. The question to ask: without this threshold information, how well can the GRU produce an output that agrees with the CONTINUOUS score produced by the threshold-based system (`anaerobic_time_frac`, `yc_test`)? Looking at the correlation between the GRU's predicted probability (`y_pred_proba`) and `yc_test` shows whether the GRU captured a threshold-like notion of effort intensity from the raw signals on its own. This counts as independent evidence of whether the model learned something physiologically meaningful rather than memorizing the label.

## 7. Save the model

**Why is the model saved in `.keras` format?**
`.keras` is Keras's own save format, which carries the architecture, weights and optimizer state in a single file, so the model can later be fully restored with a single line (`load_model`).

**Is this file enough on its own?**
No. The model's input must be scaled in the same way as the `MinMaxScaler`s saved in `gru_preprocessing.pkl` by `6_gru_preprocess.ipynb`. So in real use (e.g. serverless inference) both `gru_model.keras` and `gru_preprocessing.pkl` must be shipped together; neither produces meaningful predictions without the other.

In [ ]:
best_model.save('artifacts/gru_model.keras')
print("model saved.")